In [1]:
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import cv2, os, shutil, glob, json
import numpy as np
import pandas as pd
import torch.nn as nn
import seaborn as sns
from pathlib import Path
pd.set_option('display.max_columns', None)

In [2]:
OPTIONS = json.loads(open('../../../Task/info.json', 'r').read())
OPTIONS

{'img_size': [128, 128, 128],
 'step': 3,
 'model': 'segresnet',
 'lr': 0.0001,
 'loss': 'tversky',
 'scheduler': 'plateau'}

In [3]:
IMG_SIZE = tuple(OPTIONS.get('img_size'))
IMG_SIZE

(128, 128, 128)

In [4]:
def loadFile(path, size=(128, 128, 128)):
    file = Path(path)
    ext  = file.suffix.lower()

    if ext == '.png':
        return cv2.imread(path)
    
    if ext == '.npy':
        return np.load(path)
    
    if ext == '.dat':
        return np.reshape(np.fromfile(path, dtype=np.single), size)

    return None

def getFiles(path, limit=None, shuffle=False):
    target = sorted(glob.glob(os.path.join(path, '*')))
    if shuffle:
        np.random.shuffle(target) 
    return target[:limit]

In [5]:
images = [loadFile(img, IMG_SIZE) for img in getFiles('images')]
masks  = [loadFile(img, IMG_SIZE) for img in getFiles('masks')]

IMG_SIZE = images[0].shape[0]
print(len(images), IMG_SIZE)
images[:5]

765 128


[array([[[0.5026313 , 0.5026313 , 0.5026313 , ..., 0.4836204 ,
          0.4791169 , 0.47258404],
         [0.5026313 , 0.5026313 , 0.5026313 , ..., 0.47092772,
          0.46766356, 0.4645351 ],
         [0.5026313 , 0.5026313 , 0.5026313 , ..., 0.46536985,
          0.4653134 , 0.4623227 ],
         ...,
         [0.5026313 , 0.5026313 , 0.5026313 , ..., 0.44442868,
          0.45259112, 0.46502268],
         [0.5026313 , 0.5026313 , 0.5026313 , ..., 0.5265674 ,
          0.53611207, 0.55246   ],
         [0.5026313 , 0.5026313 , 0.5026313 , ..., 0.59355617,
          0.604294  , 0.62242967]],
 
        [[0.5026313 , 0.5026313 , 0.5026313 , ..., 0.4860953 ,
          0.48056698, 0.4722805 ],
         [0.5026313 , 0.5026313 , 0.5026313 , ..., 0.47457474,
          0.47272897, 0.46647102],
         [0.5026313 , 0.5026313 , 0.5026313 , ..., 0.46748713,
          0.4678237 , 0.46523014],
         ...,
         [0.5026313 , 0.5026313 , 0.5026313 , ..., 0.44440904,
          0.45514682, 0.

In [6]:
def setFolder(path):
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(path)

setFolder('../../../Model/Database/Target/images')
setFolder('../../../Model/Database/Target/masks')

for i, (img, mask) in enumerate(zip(images, masks)):
    np.save(f'../../../Model/Database/Target/images/img_{i:04d}.npy', img)
    np.save(f'../../../Model/Database/Target/masks/img_{i:04d}.npy', mask)